### Creating the figure for lab report modeling section ("figure or schematic that explains how one of your models works")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.append("../../code")
from clean import clean_data
from models import preprocess_for_modeling, fit_logistic_regression, LR_FEATURES
import pandas as pd

# Load, clean, preprocess, and fit
df = pd.read_csv("../../data/TBI_PUD_cleaned.csv")
df_model = preprocess_for_modeling(df)
lr_model = fit_logistic_regression(df_model)

# Pull coefficients live from fitted model
feature_labels = [
    'Altered mental status',    # cdr_ams
    'Any LOC',                  # cdr_loc_any
    'LOC > 5 sec',              # cdr_loc_over5s
    'Palpable skull fracture',  # cdr_palpable_skull_fx
    'Basilar skull fracture',   # cdr_basilar_skull_fx
    'Non-frontal hematoma',     # cdr_scalp_hematoma_nonfrontal
    'Not acting normally',      # cdr_not_acting_normally
    'Severe headache',          # cdr_severe_headache
    'Vomiting',                 # cdr_vomiting
    'Severe mechanism',         # cdr_severe_mechanism
    'Age under 2',              # is_under_2
]
coefs = lr_model.coef_[0]

# Sort by coefficient value
sorted_pairs = sorted(zip(coefs, feature_labels))
coefs_sorted, features_sorted = zip(*sorted_pairs)
colors = ['#d73027' if c > 0 else '#4575b4' for c in coefs_sorted]

fig, ax = plt.subplots(figsize=(11, 4))
ax.barh(features_sorted, coefs_sorted, color=colors, edgecolor='black', alpha=0.85)
for bar, val in zip(ax.patches, coefs_sorted):
    ax.text(val + 0.03 if val >= 0 else val - 0.03, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=6)
ax.axvline(x=0, color='black', linewidth=1.2)
ax.set_xlabel('Coefficient (log-odds of ciTBI)', fontsize=12, fontweight='bold')
ax.set_title('Logistic Regression Coefficients\nLarger positive values = stronger predictor of ciTBI',
             fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('lr_coefficients.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
dict(zip(LR_FEATURES, lr_model.coef_[0].round(3)))

# REALITY CHECK (section 3)

- Comparing my results to reality (reality is the article)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = [
    ('Total primary population',   '42,430',     '42,412',       '✓'),
    ('ciTBI rate',                  '0.89%',      '0.9%',         '✓'),
    ('ciTBI rate (under 2)',        '0.91%',      '0.86–1.13%',   '✓'),
    ('ciTBI rate (2 and older)',    '0.88%',      '0.85–0.98%',   '✓'),
    ('Neurosurgery count',          '60',         '60',           '✓'),
    ('GCS = 15 rate',               '96.8%',      '97%',          '✓'),
    ('Patients under 2',           '25.3%',      '25%',          '✓'),
    ('CT scan rate',                '35.3%',      '35.3%',        '✓'),
    ('Palpable skull fx (under 2)', '3.4%',       '3.4%',         '✓'),
    ('Palpable skull fx (2+)',      '2.1%',       '2.1%',         '✓'),
    ('Vomiting rate',               '13.1%',      '12.9–15.0%',   '✓'),
    ('AMS rate',                    '13.0%',      '11.6–13.7%',   '✓'),
]

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.axis('off')

col_labels  = ['Metric', 'Cleaned Data', 'Kuppermann et al. (2009)', 'Pass?']
col_x       = [0.01, 0.45, 0.63, 0.93]
row_height  = 0.075
y_start     = 0.92

# Header row
for j, (label, cx) in enumerate(zip(col_labels, col_x)):
    ax.text(cx, 1, label, fontsize=9, fontweight='bold',
            va='top', ha='left' if j < 3 else 'center',
            transform=ax.transAxes)

# Header underline
line = plt.Line2D([0, 1], [0.955, 0.955], transform=ax.transAxes,
                  color='black', linewidth=1.2)
ax.add_line(line)

# Data rows
for i, (metric, our_val, kup_val, status) in enumerate(metrics):
    y = y_start - i * row_height
    bg = '#f0f4f8' if i % 2 == 0 else 'white'
    ax.add_patch(plt.Rectangle((0, y - row_height * 0.82), 1, row_height * 0.9,
                                transform=ax.transAxes, color=bg, zorder=0))
    ax.text(col_x[0], y, metric,  fontsize=8.2, va='top', ha='left',   transform=ax.transAxes)
    ax.text(col_x[1], y, our_val, fontsize=8.2, va='top', ha='left',   transform=ax.transAxes)
    ax.text(col_x[2], y, kup_val, fontsize=8.2, va='top', ha='left',   transform=ax.transAxes)
    ax.text(col_x[3], y, status,  fontsize=9.5, va='top', ha='center',
            transform=ax.transAxes, color='#1a7a1a', fontweight='bold')

# Bottom line
bot_y = y_start - len(metrics) * row_height + 0.01
bottom_line = plt.Line2D([0, 1], [bot_y, bot_y], transform=ax.transAxes,
                          color='black', linewidth=0.8)
ax.add_line(bottom_line)

ax.set_title('Reality Check: Cleaned Data vs. Kuppermann et al. (2009)',
             fontsize=10.5, fontweight='bold', pad=8)

plt.tight_layout()

plt.savefig('reality_check_table.png', dpi=300, bbox_inches='tight', facecolor='white')

plt.show()

# STABILITY CHECK (on perturbed data)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df_orig = pd.read_csv("../../data/TBI_PUD_cleaned.csv")
df_pert = pd.read_csv("../../data/TBI_PUD_cleaned_perturbed.csv")

orig = df_orig[df_orig['in_primary_analysis'] == 1]
pert = df_pert[df_pert['in_primary_analysis'] == 1]

gcs_total = pd.to_numeric(df['gcs_total_score'], errors='coerce')

def compute_finding2_rr(df_pop):
    age = pd.to_numeric(df_pop['patient_age_years'], errors='coerce')
    age_groups = [('<2 yr', 0, 2), ('2-5 yr', 2, 6), ('6-12 yr', 6, 13), ('13-17 yr', 13, 18)]
    skull_rr, vom_rr = [], []
    for label, lo, hi in age_groups:
        subset = df_pop[(age >= lo) & (age < hi)]
        for col, lst in [('has_palpable_skull_fracture', skull_rr), 
                          ('has_vomiting_post_injury', vom_rr)]:
            pos = subset[pd.to_numeric(subset[col], errors='coerce') == 1]
            neg = subset[pd.to_numeric(subset[col], errors='coerce') == 0]
            citbi = 'has_clinically_important_tbi'
            r_pos = (pd.to_numeric(pos[citbi], errors='coerce')==1).sum()/len(pos) if len(pos)>0 else 0
            r_neg = (pd.to_numeric(neg[citbi], errors='coerce')==1).sum()/len(neg) if len(neg)>0 else 0
            lst.append(round(r_pos/r_neg, 1) if r_neg > 0 else 0)
    return skull_rr, vom_rr

# orig = df[df['in_primary_analysis'] == 1]
# pert = df[gcs_total >= 13]
skull_orig, vom_orig = compute_finding2_rr(orig)
skull_pert, vom_pert = compute_finding2_rr(pert)

age_labels = ['<2 yr', '2-5 yr', '6-12 yr', '13-17 yr']
x = np.arange(len(age_labels))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
fig.suptitle('Stability Check: Finding 2 Under GCS Threshold Perturbation\n'
             'Before: GCS $\\geq$14 (primary population)  |  After: GCS $\\geq$13 (expanded population)',
             fontsize=11, fontweight='bold', y=1.02)

colors_orig = ['#c0392b', '#e74c3c', '#f1948a', '#fadbd8']
colors_pert = ['#7f8c8d', '#95a5a6', '#bdc3c7', '#ecf0f1']

for ax, orig_rr, pert_rr, title, colors_o in [
    (axes[0], skull_orig, skull_pert, 'Palpable Skull Fracture', colors_orig),
    (axes[1], vom_orig, vom_pert, 'Vomiting', ['#1e8449','#27ae60','#58d68d','#d5f5e3'])
]:
    bars1 = ax.bar(x - width/2, orig_rr, width, label='Original (GCS ≥14)', 
                   color=colors_o, edgecolor='black', linewidth=0.8)
    bars2 = ax.bar(x + width/2, pert_rr, width, label='Perturbed (GCS ≥13)',
                   color=colors_pert, edgecolor='black', linewidth=0.8, hatch='//')
    
    for bar, val in zip(bars1, orig_rr):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val}x', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    for bar, val in zip(bars2, pert_rr):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val}x', ha='center', va='bottom', fontsize=8.5, color='#555')
    
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(age_labels)
    ax.set_ylabel('Risk Ratio (presence vs. absence)', fontsize=9)
    ax.set_xlabel('Age Group', fontsize=9)
    ax.legend(fontsize=8.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, max(max(orig_rr), max(pert_rr)) * 1.25)

plt.tight_layout()
plt.savefig('stability_check_finding2.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(df_orig['in_primary_analysis'].value_counts())
print(df_pert['in_primary_analysis'].value_counts())

# Stability check for MODELING section

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ['CDR', 'Logistic\nRegression', 'Random\nForest']

orig_sens = [0.9681, 0.7686, 0.9495]
pert_sens = [0.9718, 0.7700, 0.9484]

orig_spec = [0.5658, 0.8410, 0.6547]
pert_spec = [0.5624, 0.8465, 0.6673]

x = np.arange(len(models))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Stability Check: Model Performance Under GCS Threshold Perturbation\n'
             'Before: GCS $\\geq$14 (n=42,412)  |  After: GCS $\\geq$13 (n=42,716)',
             fontsize=11, fontweight='bold', y=1.02)

colors_orig = ['#2471a3', '#1a5276', '#154360']
colors_pert = ['#7f8c8d', '#95a5a6', '#bdc3c7']

for ax, orig_vals, pert_vals, title, ylabel in [
    (axes[0], orig_sens, pert_sens, 'Sensitivity', 'Sensitivity'),
    (axes[1], orig_spec, pert_spec, 'Specificity', 'Specificity'),
]:
    bars1 = ax.bar(x - width/2, orig_vals, width, label='Original (GCS ≥14)',
                   color=colors_orig, edgecolor='black', linewidth=0.8)
    bars2 = ax.bar(x + width/2, pert_vals, width, label='Perturbed (GCS ≥13)',
                   color=colors_pert, edgecolor='black', linewidth=0.8, hatch='//')

    for bar, val in zip(bars1, orig_vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.1%}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    for bar, val in zip(bars2, pert_vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.1%}', ha='center', va='bottom', fontsize=8.5, color='#555')

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_ylim(0, min(1.0, max(max(orig_vals), max(pert_vals)) * 1.18))
    ax.legend(fontsize=8.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('stability_check_modeling.png', dpi=300, bbox_inches='tight', facecolor='white')